<div class="cover">

# Lab 6 — Deep Learning for NLP

**Name:** Hadi Muneer Abu Allairat<br>
**Student ID:** 2230005761<br>
**Dataset:** `07-yelp-dataset.txt` — 1,000 labelled Yelp restaurant reviews

</div>

The lab builds a binary sentiment classifier on Yelp reviews with PyTorch. The walkthrough uses
TF-IDF features and a two-hidden-layer network; the two tasks replace the features with averaged
Word2Vec vectors and the network with a deeper one.

Since the point of the tasks is to compare two representations, I run the TF-IDF baseline first under
the same conditions so the comparison is meaningful, then change one thing at a time.

In [1]:
import random
import re
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()
print("torch", torch.__version__)

torch 2.6.0+cu124


## 1. Load the Yelp dataset

Tab-separated, no header row: the review text and a 0/1 sentiment label.

In [2]:
df = pd.read_csv("dataset/07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()

Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [3]:
print(df["label"].value_counts())
print()
print("missing values:", df.isna().sum().sum())
print("duplicate reviews:", df["review"].duplicated().sum())
print("median review length:", int(df["review"].str.split().str.len().median()), "words")

label
1    500
0    500
Name: count, dtype: int64

missing values: 0
duplicate reviews: 4
median review length: 10 words


Exactly 500 of each class. A perfectly balanced dataset means accuracy is a usable headline metric
here — the majority-class baseline is 50%, so anything meaningfully above that is real signal. That
is not the usual situation, and it is worth noting explicitly rather than assuming.

The dataset is also small: 1,000 reviews, a median of 10 words each. That size drives most of what
follows.

## 2. Preprocessing

In [4]:
def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    # 3. Collapse runs of whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


print("before:", df["review"].iloc[0])
print("after: ", preprocess_text(df["review"].iloc[0]))

before: Wow... Loved this place.
after:  wow loved this place


In [5]:
df["review"] = df["review"].apply(preprocess_text)
df.head()

,review,label
0,wow loved this place,1
1,crust is not good,0
2,not tasty and the texture was just nasty,0
3,stopped by during the late may bank holiday of...,1
4,the selection on the menu was great and so wer...,1


A note on the version of this cell in the lab sheet: the two lines that apply the function are
indented *inside* `preprocess_text`, below its `return`. Python never reaches them, so the function
is defined, nothing is applied, and the cell runs without error — the model then trains on raw
untouched text. It is a silent bug rather than a crash, which is what makes it worth flagging. The
`apply` is pulled out to module level above.

## 3. Train/test split

`stratify=y` keeps the 50/50 class balance in both halves. The lab's split omits it; on a dataset
this small, a random 80/20 split can drift several points away from balanced by chance, which would
make the test accuracy harder to interpret.

In [6]:
X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print("Training reviews:", len(X_train))
print("Testing reviews: ", len(X_test))
print("train balance:", y_train.value_counts().to_dict())
print("test balance: ", y_test.value_counts().to_dict())

Training reviews: 800
Testing reviews:  200
train balance: {1: 400, 0: 400}
test balance:  {1: 100, 0: 100}


## 4. Baseline — TF-IDF features

In [7]:
tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape: ", X_test_tfidf.shape)
print("vocabulary kept:", len(tfidf.vocabulary_))

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape:  (200, 1000)
vocabulary kept: 1000


`fit_transform` on train and plain `transform` on test is the part that matters. Fitting the
vectoriser on the full dataset would let the test set influence the vocabulary and the IDF weights,
which inflates the score without improving the model.

In [8]:
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

print(X_train_tensor.shape, y_train_tensor.shape)

torch.Size([800, 1000]) torch.Size([800, 1])


The labels are reshaped to a column. `BCEWithLogitsLoss` compares the model output, shape
`(n, 1)`, against the targets elementwise; a flat `(n,)` target would broadcast into an `(n, n)`
comparison and silently train on nonsense.

## 5. The training loop

One function, reused for every model below so the comparison is not confounded by differences in the
training procedure.

In [9]:
def train_model(model, X, y, epochs, lr, log_every=5):
    loss_function = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = []
    for epoch in range(epochs):
        model.train()

        optimizer.zero_grad()          # see note below
        outputs = model(X)
        loss = loss_function(outputs, y)
        loss.backward()
        optimizer.step()

        history.append(loss.item())
        if (epoch + 1) % log_every == 0:
            print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

    return history


@torch.no_grad()
def evaluate(model, X, y_true, name):
    model.eval()
    probabilities = torch.sigmoid(model(X))
    predictions = (probabilities >= 0.5).int().numpy().flatten()

    accuracy = accuracy_score(y_true, predictions)
    print(f"{name} test accuracy: {accuracy:.3f}")
    return accuracy, predictions

`optimizer.zero_grad()` is the other correction to the lab sheet, and it is the one that changes the
numbers. PyTorch *accumulates* gradients into `.grad` rather than overwriting them, so without this
call, epoch 3 steps on the sum of the gradients from epochs 1, 2 and 3. The effective learning rate
grows every epoch. The loss usually still falls, which is exactly why the bug survives — it looks
like it is working.

I verify this rather than assert it: the same model is trained both ways further down.

## 6. Baseline network

Two hidden layers, as in the walkthrough.

In [10]:
set_seed()

baseline = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
)

print(baseline)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


In [11]:
baseline_history = train_model(baseline, X_train_tensor, y_train_tensor, epochs=10, lr=0.01)
tfidf_accuracy, tfidf_predictions = evaluate(baseline, X_test_tensor, y_test, "TF-IDF baseline")

Epoch 5/10 - Loss: 0.6242
Epoch 10/10 - Loss: 0.3577
TF-IDF baseline test accuracy: 0.755


In [12]:
print("Classification Report:")
print(classification_report(y_test, tfidf_predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(pd.DataFrame(
    confusion_matrix(y_test, tfidf_predictions),
    index=["actual neg", "actual pos"],
    columns=["pred neg", "pred pos"],
))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.81      0.67      0.73       100
    Positive       0.72      0.84      0.77       100

    accuracy                           0.76       200
   macro avg       0.76      0.76      0.75       200
weighted avg       0.76      0.76      0.75       200

Confusion Matrix:
            pred neg  pred pos
actual neg        67        33
actual pos        16        84


### Checking the `zero_grad` claim

Same architecture, same seed, same data — the only difference is whether gradients are cleared.

In [13]:
def train_without_zero_grad(model, X, y, epochs, lr):
    loss_function = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        model.train()
        outputs = model(X)
        loss = loss_function(outputs, y)
        loss.backward()        # gradients accumulate across epochs
        optimizer.step()
    return loss.item()


set_seed()
buggy = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1),
)
buggy_loss = train_without_zero_grad(buggy, X_train_tensor, y_train_tensor, epochs=10, lr=0.01)
buggy_accuracy, _ = evaluate(buggy, X_test_tensor, y_test, "without zero_grad")

print(f"\nfinal training loss   with zero_grad: {baseline_history[-1]:.4f}")
print(f"                   without zero_grad: {buggy_loss:.4f}")
print(f"test accuracy         with zero_grad: {tfidf_accuracy:.3f}")
print(f"                   without zero_grad: {buggy_accuracy:.3f}")

without zero_grad test accuracy: 0.740

final training loss   with zero_grad: 0.3577
                   without zero_grad: 0.3854
test accuracy         with zero_grad: 0.755
                   without zero_grad: 0.740


## 7. Task 1 — Represent the reviews with Word2Vec

Word2Vec gives a vector per *word*, but the classifier needs one vector per *review*. The standard
cheap solution is to average the word vectors, which is what I do here.

Two decisions worth stating:

**The model is trained on the training reviews only.** Training it on all 1,000 would let the test
reviews shape the embedding space, which is leakage — the same reason the TF-IDF vectoriser was fit
on the training half alone.

**Reviews with no known words get a zero vector.** With `min_count=1` every training word is in the
vocabulary, but a test review can consist entirely of words never seen in training. Averaging an
empty list would produce `nan` and poison the loss, so those fall back to zeros.

In [14]:
import gensim

train_tokens = [review.split() for review in X_train]
test_tokens = [review.split() for review in X_test]

w2v = gensim.models.Word2Vec(
    train_tokens,
    vector_size=200,
    window=5,
    min_count=1,
    sg=1,
    seed=SEED,
    workers=1,   # reproducibility: multi-threaded training is nondeterministic
)

print(w2v)
print("vocabulary size:", len(w2v.wv))

Word2Vec<vocab=1787, vector_size=200, alpha=0.025>
vocabulary size: 1787


In [15]:
def document_vector(tokens, model):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if not vectors:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0)


X_train_w2v = np.vstack([document_vector(t, w2v) for t in train_tokens])
X_test_w2v = np.vstack([document_vector(t, w2v) for t in test_tokens])

print("train:", X_train_w2v.shape)
print("test: ", X_test_w2v.shape)

empty = sum(1 for t in test_tokens if not any(w in w2v.wv for w in t))
oov = sum(1 for t in test_tokens for w in t if w not in w2v.wv)
total = sum(len(t) for t in test_tokens)
print(f"\ntest tokens unseen in training: {oov}/{total} ({oov / total:.1%})")
print("test reviews with no known words:", empty)

train: (800, 200)
test:  (200, 200)

test tokens unseen in training: 281/2157 (13.0%)
test reviews with no known words: 0


In [16]:
X_train_w2v_tensor = torch.FloatTensor(X_train_w2v)
X_test_w2v_tensor = torch.FloatTensor(X_test_w2v)

print(X_train_w2v_tensor.shape, X_train_w2v_tensor.dtype)
print("any nan:", torch.isnan(X_train_w2v_tensor).any().item())

torch.Size([800, 200]) torch.float32
any nan: False


The feature count drops from 1,000 TF-IDF columns to 200 dense dimensions — a fifth of the width,
with every dimension active rather than a mostly-zero sparse row.

## 8. Task 2 — A deeper network on the Word2Vec features

Input → 128 → 64 → 32 → 1, ReLU between layers, `BCEWithLogitsLoss`, Adam at `lr=0.001`, 15 epochs.

No `Sigmoid` on the output layer: `BCEWithLogitsLoss` applies it internally in a numerically stable
way, and adding it here would apply it twice.

In [17]:
set_seed()

deep_model = nn.Sequential(
    nn.Linear(X_train_w2v_tensor.shape[1], 128),
    nn.ReLU(),

    nn.Linear(128, 64),
    nn.ReLU(),

    nn.Linear(64, 32),
    nn.ReLU(),

    nn.Linear(32, 1),
)

print(deep_model)
print("trainable parameters:", sum(p.numel() for p in deep_model.parameters()))

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)
trainable parameters: 36097


In [18]:
history = train_model(
    deep_model, X_train_w2v_tensor, y_train_tensor, epochs=15, lr=0.001, log_every=3
)

w2v_accuracy, w2v_predictions = evaluate(deep_model, X_test_w2v_tensor, y_test, "Word2Vec + deep net")

Epoch 3/15 - Loss: 0.6949
Epoch 6/15 - Loss: 0.6941
Epoch 9/15 - Loss: 0.6935
Epoch 12/15 - Loss: 0.6933
Epoch 15/15 - Loss: 0.6932
Word2Vec + deep net test accuracy: 0.490


In [19]:
print("Classification Report:")
print(classification_report(y_test, w2v_predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(pd.DataFrame(
    confusion_matrix(y_test, w2v_predictions),
    index=["actual neg", "actual pos"],
    columns=["pred neg", "pred pos"],
))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.49      0.98      0.66       100
    Positive       0.00      0.00      0.00       100

    accuracy                           0.49       200
   macro avg       0.25      0.49      0.33       200
weighted avg       0.25      0.49      0.33       200

Confusion Matrix:
            pred neg  pred pos
actual neg        98         2
actual pos       100         0


In [20]:
print(f"first epoch loss: {history[0]:.4f}")
print(f"last epoch loss:  {history[-1]:.4f}")
print(f"total decrease:   {history[0] - history[-1]:.4f}")

first epoch loss: 0.6958
last epoch loss:  0.6932
total decrease:   0.0026


The required settings produce a model that has not learned anything: the loss moves by 0.003 across
all 15 epochs and sits at 0.693, which is `-ln(0.5)` — the loss of a model outputting 50/50 for every
input. The classification report confirms it, with 0.00 recall on the positive class. It is
predicting "negative" for almost everything and scoring near 50% because the test set is balanced.

That is the answer to Task 2 as specified. But "the model failed" is not a diagnosis, and there are
two candidate explanations worth separating: the Word2Vec features carry no signal, or the model was
not trained long enough to use them.

## 9. Why did it fail?

The training loop here is **full-batch**: one forward pass, one backward pass, one weight update per
epoch. So 15 epochs is **15 weight updates**, at `lr=0.001`. The TF-IDF baseline got 10 updates at
`lr=0.01` — ten times the step size. The two models were never given comparable training budgets.

A second candidate is feature scale. TF-IDF rows are L2-normalised to unit length, while averaged
Word2Vec vectors have whatever magnitude the embedding produces; if that were much smaller, the
gradients reaching the first layer would be small too. That is worth measuring rather than assuming.

In [21]:
print("mean row norm, TF-IDF features:  ", float(X_train_tensor.norm(dim=1).mean()))
print("mean row norm, Word2Vec features:", float(X_train_w2v_tensor.norm(dim=1).mean()))
print()
print("updates given to TF-IDF baseline: 10 at lr=0.01")
print("updates given to Word2Vec model:  15 at lr=0.001")

mean row norm, TF-IDF features:   1.0
mean row norm, Word2Vec features: 1.0421229600906372

updates given to TF-IDF baseline: 10 at lr=0.01
updates given to Word2Vec model:  15 at lr=0.001


The scale explanation is **ruled out**: the mean row norm is 1.00 for TF-IDF and 1.04 for the
averaged Word2Vec vectors. They are already on the same scale, so normalisation would change nothing
and the small gradients have to come from somewhere else.

That leaves the training budget. To test it directly: the same architecture and the same features,
trained for 500 epochs instead of 15. Nothing else changes.

In [22]:
set_seed()

longer_model = nn.Sequential(
    nn.Linear(X_train_w2v_tensor.shape[1], 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1),
)

long_history = train_model(
    longer_model, X_train_w2v_tensor, y_train_tensor, epochs=500, lr=0.001, log_every=100
)

w2v_long_accuracy, w2v_long_predictions = evaluate(
    longer_model, X_test_w2v_tensor, y_test, "Word2Vec + deep net, 500 epochs"
)

Epoch 100/500 - Loss: 0.6889


Epoch 200/500 - Loss: 0.6468


Epoch 300/500 - Loss: 0.6044


Epoch 400/500 - Loss: 0.5933


Epoch 500/500 - Loss: 0.5779
Word2Vec + deep net, 500 epochs test accuracy: 0.675


In [23]:
print("Classification Report:")
print(classification_report(y_test, w2v_long_predictions,
                            target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(pd.DataFrame(
    confusion_matrix(y_test, w2v_long_predictions),
    index=["actual neg", "actual pos"],
    columns=["pred neg", "pred pos"],
))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.72      0.57      0.64       100
    Positive       0.64      0.78      0.71       100

    accuracy                           0.68       200
   macro avg       0.68      0.68      0.67       200
weighted avg       0.68      0.68      0.67       200

Confusion Matrix:
            pred neg  pred pos
actual neg        57        43
actual pos        22        78


## 10. Comparison

In [24]:
pd.DataFrame({
    "features": ["TF-IDF (1000 sparse)",
                 "Word2Vec avg (200 dense)",
                 "Word2Vec avg (200 dense)"],
    "hidden layers": ["64, 32", "128, 64, 32", "128, 64, 32"],
    "epochs": [10, 15, 500],
    "lr": [0.01, 0.001, 0.001],
    "test accuracy": [round(tfidf_accuracy, 3),
                      round(w2v_accuracy, 3),
                      round(w2v_long_accuracy, 3)],
    "note": ["walkthrough", "Task 2 as specified", "same model, longer training"],
})

,features,hidden layers,epochs,lr,test accuracy,note
0,TF-IDF (1000 sparse),"64, 32",10,0.010,0.755,walkthrough
1,Word2Vec avg (200 dense),"128, 64, 32",15,0.001,0.490,Task 2 as specified
2,Word2Vec avg (200 dense),"128, 64, 32",500,0.001,0.675,"same model, longer training"


## Findings

**Task 2 as specified scores 0.490 — no better than guessing.** Taken at face value that looks like a
verdict on Word2Vec. It is not, and the diagnostic separates the two causes.

**Most of the gap was undertraining.** Holding everything else fixed and training the same model for
500 epochs instead of 15 lifts it from **0.490 to 0.675**. The specified budget of 15 full-batch
epochs is 15 weight updates at `lr=0.001`; the model never left its initialisation, which is why the
loss sat at 0.693 — exactly `-ln(0.5)` — and every prediction came out "negative". The lesson is that
epoch count is not comparable across configurations: the TF-IDF baseline reached 0.755 in 10 epochs
only because its learning rate was ten times larger. Comparing two models on equal epochs is not
comparing them on equal training.

I expected feature scale to be part of it and measured it instead of assuming: both representations
have a mean row norm of ~1.0, so that explanation is ruled out.

**The remaining gap — 0.675 against 0.755 — is the representation, and it is real.** Two reasons:

- Word2Vec learns from co-occurrence statistics, and 800 reviews of ~10 words each is roughly 8,000
  training tokens against the billions behind published embeddings. With `min_count=1`, most of the
  1,787 words here are seen a handful of times, so their vectors stay near random initialisation.
- Averaging discards word order and lets function words dilute the sentiment-bearing ones. *"the food
  was not good"* and *"the food was good"* differ by one short word that barely moves the mean, while
  TF-IDF gives `not` its own dimension and its own weight.

So the fair conclusion is narrower than "TF-IDF wins": on a dataset this small, *training Word2Vec
from scratch* loses to TF-IDF. The realistic fix is pretrained vectors (GoogleNews, GloVe), using
these reviews only to train the classifier on top.

Two corrections to the lab sheet were needed before any of these numbers could be trusted: the
preprocessing function whose `apply` lines sit below its `return` and therefore never run, and the
missing `optimizer.zero_grad()` (worth 0.755 against 0.740 here). Both fail silently — the notebook
runs, the loss falls, and the output looks reasonable.